# B1：IDM 跟驰模型参数标定
# B1: IDM Car-Following Model Calibration

## 实验目标 | Objectives
- 理解 IDM（Intelligent Driver Model）跟驰模型的物理意义
  Understand the IDM car-following model
- 用 SUMO 真实数据标定 IDM 参数
  Calibrate IDM parameters from real SUMO data

## 背景知识 | Background
**IDM**：Treiber 提出的跟驰模型，认为驾驶员加速度取决于
与前车的间距、速度差。公式：
`a = a_max * [1 - (v/v0)^4 - (s*/s)^2]`
其中 `s*` 是期望间距（与速度、速度差相关）。

**参数标定**：SUMO 默认用一套 IDM 参数，但真实驾驶员风格各异。
本实验从 SUMO 采集真实跟驰数据，用最小二乘反推 `a`（最大加速度）、
`b`（舒适减速度），这是交通工程的常用方法。


In [ ]:
import numpy as np
print('✓ 导入成功')


## 1. IDM 模型定义

In [ ]:
def idm_acceleration(v, v0, gap, dv, a=1.0, b=1.5, s0=2.0, T=1.0):
    """IDM 加速度计算
    v: 自车速度, v0: 期望速度, gap: 间距, dv: 速度差（自车-前车）
    """
    s_star = s0 + v*T + v*dv/(2*np.sqrt(a*b))
    return a * (1 - (v/v0)**4 - (s_star/max(gap, 0.1))**2)
print('✓ IDM 模型定义完成')


## 2. 从 SUMO 采集真实跟驰数据

In [ ]:
import os
workdir = '/tmp/b1_sumo'
os.makedirs(workdir, exist_ok=True)
os.chdir(workdir)
os.system('netgenerate --grid --grid.x-number=3 --grid.y-number=1 --grid.length=1000 --output-file=grid.net.xml 2>&1 | tail -1')
with open('routes.rou.xml', 'w') as f:
    f.write('''<routes>
    <vType id="car" carFollowModel="IDM" accel="1.5" decel="2.0" maxSpeed="25"/>
    <route id="r0" edges="A0B0 B0C0"/>
    <vehicle id="lead" type="car" route="r0" depart="0" departPos="1000" departSpeed="15"/>
    <vehicle id="follow" type="car" route="r0" depart="0" departPos="970" departSpeed="15"/>
</routes>''')
with open('sumo.sumocfg', 'w') as f:
    f.write('<configuration><input><net-file value="grid.net.xml"/><route-files value="routes.rou.xml"/></input></configuration>')

import traci
traci.start(['sumo', '-c', 'sumo.sumocfg', '--no-warnings', '--no-step-log'])
data = []  # (v, gap, dv, a_real)
for step in range(300):
    traci.simulationStep()
    try:
        v = traci.vehicle.getSpeed('follow')
        v_lead = traci.vehicle.getSpeed('lead')
        x = traci.vehicle.getPosition('follow')[0]
        x_lead = traci.vehicle.getPosition('lead')[0]
        gap = x_lead - x - 4.5
        a = traci.vehicle.getAcceleration('follow')
        data.append((v, gap, v - v_lead, a))
    except:
        continue
traci.close()
print(f'✓ 采集 {len(data)} 个数据点（真 SUMO IDM）')


## 3. 参数标定（最小二乘）

In [ ]:
from scipy.optimize import minimize
data = np.array(data)
v, gap, dv, a_real = data[:, 0], data[:, 1], data[:, 2], data[:, 3]

def loss(params):
    a_idm, b_idm = params
    a_pred = np.array([idm_acceleration(vi, 25, gi, dvi, a=a_idm, b=b_idm)
                       for vi, gi, dvi in zip(v, gap, dv)])
    return np.sqrt(np.mean((a_pred - a_real)**2))

res = minimize(loss, x0=[1.0, 1.5], bounds=[(0.5, 3.0), (0.5, 3.0)])
a_opt, b_opt = res.x
rmse = res.fun
print(f'标定结果: a={a_opt:.2f}, b={b_opt:.2f}, RMSE={rmse:.3f}')


## 4. 评测断言

In [ ]:
assert rmse < 2.0, f'RMSE {rmse:.2f} 超标!'
print('✅ 评测通过！IDM 参数标定成功')


## 🏆 挑战任务
1. 标定 T（期望时距）和 s0（最小间距）。
2. 对比不同驾驶风格（激进/保守）的参数差异。